# 🎙️ Chatterbox TTS — Kaggle 2× T4 GPU

**Resemble AI's Chatterbox Multilingual V3** (the checkpoint behind the
[official V3 Space](https://huggingface.co/spaces/ResembleAI/Chatterbox-Multilingual-TTS-V3)) — 23 languages,
zero-shot voice cloning — served as a **phone-sized web app** from a Kaggle **GPU T4 x2** runtime.

Everything stays in your Kaggle session; a Cloudflare tunnel carries it to your phone.

---

### 📲 How to run this (from your phone)

1. **Settings → Accelerator: `GPU T4 x2`, Internet: `On`**, then open the notebook.
2. **Tap ▶ on each numbered step**, top to bottom. Code stays collapsed into slim
   one-line bars after **Step 0** — *tap a bar* if you ever want to peek.
3. **Step 8 prints a big tappable `trycloudflare.com` URL** → open it, then *Add to
   Home Screen* and use it like an app.
4. First run downloads ~5 GB of weights (~4-6 min); they cache under
   `/kaggle/working`, so later sessions skip the download.

<details><summary><b>ℹ️ What this notebook does under the hood (tap to expand)</b></summary>

* **Custom mobile SPA instead of the Gradio Space** — the official Space caps text at 300 chars and renders a desktop
  grid; here you get sticky-thumb Generate, bottom-sheet language & voice pickers, clone-a-voice upload from your
  gallery/recorder, PWA install, and real SSE progress.
* **Up to two replicas per GPU (four total)** + a **work-stealing chunk queue** → up to four text parts render at once. Replicas are loaded sequentially and an OOM automatically leaves that GPU at its last safe count.
* **EPUB importer** — upload a book, inspect its chapter list, select chapters, and create a separately named audio job for every title.
* **Optional Google Drive mirroring** — every completed render is first saved under `/kaggle/working/chatterbox_outputs`, then mirrored to Drive when configured; a phone/tunnel disconnect cannot cancel it.
* **FP16 autocast with a per-GPU self-test** (auto-fallback to FP32), TF32 + cuDNN autotune.
* **Voice conditionals cached per GPU per clip** — reference embedding runs once, not per chunk.
* **Pipelined CPU encoding** (wav/mp3/opus/flac) and parallel replica loading threads.
* Full tuning parity with the Space: exaggeration, CFG/pace, temperature, seed (deterministic per chunk), plus
  advanced min_p / top_p / repetition-penalty / gap / chunk-size controls.

</details>


### 0️⃣ Phone mode · ~1 s

Run this first: it injects a tiny stylesheet that **collapses every code cell into a slim bar** across the page —
tap a bar to peek or edit, it springs back after running. Cosmetic only; everything works the same.

<details><summary>ℹ️ How it works / how to revert</summary>

Pure CSS on the CodeMirror containers (`max-height` + fade + `:focus-within` expand), so it survives Kaggle's output
sanitiser. If the browser allows output JavaScript, a floating **☰ chip** also appears for *show/hide all code*.
To revert the classic look: ⋮ on this cell → **Clear output**.

</details>


In [ ]:
# ── STEP 0 · 📱 Phone mode: collapse all code cells into slim tap-to-peek bars ──
# Pure CSS, so it survives Kaggle's output sanitiser; a floating ☰ chip is added
# when output JavaScript is allowed. Revert any time: clear this cell's output.
from IPython.display import display, HTML

COMPACT_CSS = r"""
<style id="cbphone">
/* CodeMirror 6 (current JupyterLab + Kaggle) & CodeMirror 5 (classic):
   clip editors to a one-line peek bar with a fade */
html:not(.cbshowall) .cm-editor,
html:not(.cbshowall) .CodeMirror {
  max-height: 2.9em !important;
  overflow: hidden !important;
  border-radius: 10px !important;
  opacity: .85;
  -webkit-mask-image: linear-gradient(to bottom, #000 40%, transparent 95%);
          mask-image: linear-gradient(to bottom, #000 40%, transparent 95%);
}
/* tap the bar -> the editor gets focus -> expand fully until it loses focus */
html:not(.cbshowall) .cm-editor:focus-within,
html:not(.cbshowall) .CodeMirror:focus-within {
  max-height: none !important;
  opacity: 1;
  -webkit-mask-image: none; mask-image: none;
}
@media (max-width: 760px) {
  .cm-editor, .CodeMirror { font-size: 12.5px !important; }
  div.prompt, .jp-InputPrompt { font-size: 10px !important; }
}
#cbchip {
  position: fixed; right: 12px; bottom: 88px; z-index: 2147483000;
  background: #1f1b2b; color: #a78bfa; border: 1px solid #2b2540;
  border-radius: 999px; padding: 11px 15px;
  font: 600 13px/1 -apple-system, system-ui, sans-serif;
  box-shadow: 0 8px 24px rgba(0,0,0,.4); -webkit-user-select: none; user-select: none;
  cursor: pointer;
}
</style>
"""

CHIP_JS = r"""
(function () {
  try {
    if (document.getElementById('cbchip')) return;
    var chip = document.createElement('div');
    chip.id = 'cbchip';
    chip.textContent = '☰ code: bars';
    chip.onclick = function () {
      var shown = document.documentElement.classList.toggle('cbshowall');
      chip.textContent = shown ? '☰ code: shown' : '☰ code: bars';
    };
    document.body.appendChild(chip);
  } catch (e) {}
})();
"""

display(HTML(COMPACT_CSS))
try:
    from IPython.display import Javascript
    display(Javascript(CHIP_JS))
except Exception:
    pass   # sanitised on some frontends — the CSS above is still active

display(HTML(
    '<div style="margin:2px 0;padding:10px 12px;border-radius:12px;'
    'background:#17141f;border:1px solid #2b2540;color:#a49eb8;'
    'font:13px/1.45 -apple-system,system-ui,sans-serif">'
    '📱 <b style="color:#a78bfa">Phone mode on.</b> Code cells are now slim bars — '
    '<b style="color:#eeeaf6">tap one</b> to peek or edit, it collapses after running. '
    'Look for the <b style="color:#eeeaf6">☰ chip</b> to show everything at once, or '
    'clear this cell’s output to revert.</div>'))
print("phone mode: on")


### 1️⃣ Install everything · ~3 min (first run)

Chatterbox from GitHub `@master` — with the **V3** checkpoint support the PyPI wheel lacks — plus its pinned deps and
the web-server bits. Kaggle's own torch/CUDA stays untouched.

<details><summary>ℹ️ Note on the install strategy</summary>

*`--no-deps` + hand-picked pins:* master pins `torch==2.6.0`; Kaggle ships its own CUDA-matched torch/torchaudio and
we keep it. We also skip master's `gradio` pin (custom UI below). `transformers==5.2.0` matches upstream master.

</details>


In [ ]:
# ── STEP 1 · Install everything (~3 min first run) ──────────────────────

import sys, subprocess, os

def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

# --- Chatterbox itself -------------------------------------------------------
# The PyPI wheel (0.1.7) still ships the V2 multilingual checkpoint only.
# GitHub master adds `t3_model="v3"` support (the checkpoint the official
# "Chatterbox-Multilingual-TTS-V3" Space serves), so we install from source.
# --no-deps: DO NOT let it touch torch/torchaudio — Kaggle's CUDA builds stay put.
pip("--no-deps", "chatterbox-tts @ git+https://github.com/resemble-ai/chatterbox.git@master")

# --- Upstream-pinned runtime deps (minus torch/torchaudio/gradio) ------------
# transformers==5.2.0 matches upstream master; gradio is skipped on purpose
# (we serve our own mobile UI below).
pip("transformers==5.2.0", "librosa==0.11.0", "s3tokenizer", "diffusers==0.29.0",
    "conformer==0.3.2", "safetensors==0.5.3", "spacy-pkuseg", "pykakasi==2.3.0",
    "pyloudnorm", "omegaconf", "einops",
    "resemble-perth @ git+https://github.com/resemble-ai/Perth.git@master")

# --- Notebook-side deps ------------------------------------------------------
pip("fastapi>=0.115", "uvicorn[standard]>=0.30", "pydub", "python-multipart",
    "soundfile", "hf_transfer", "EbookLib", "beautifulsoup4", "google-api-python-client", "google-auth")

# System-level: ffmpeg (mp3/opus encode + voice-upload transcoding)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)

# Cloudflared for the public tunnel (self-contained binary, no login required)
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared",
    ])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

print("deps ok")


### 2️⃣ Detect the GPUs · ~1 s

You should see **2× Tesla T4 (16 GB)** listed. One full Chatterbox replica loads per GPU — a genuine ~2× wall-clock
speedup on long scripts. Single-GPU or CPU runtimes still work, just slower; it also enables TF32 + cuDNN autotune.


In [ ]:
# ── STEP 2 · Detect GPUs & set performance flags ─────────────────────────

import os, multiprocessing as mp

# --- ENV must be set before torch/huggingface_hub are imported ---------------
# Persist the HF cache under /kaggle/working so the ~5 GB of Chatterbox weights
# survive across sessions of the same Kaggle project (20 GB quota).
os.environ.setdefault("HF_HOME", "/kaggle/working/.cache/huggingface")
# Multi-threaded (Rust) downloader — the T3 safetensors alone is ~2 GB.
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # we manage our own threads

import torch

# --- Free speed -----------------------------------------------------------
# TF32 matmuls on Ampere+ (ignored gracefully on the T4's older arch, but it
# also speeds up cuDNN fp32 convs; harmless if unsupported).
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True     # fixed-ish shapes per chunk → autotuned convs

N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPUS = []
for _i in range(N_GPU):
    _p = torch.cuda.get_device_properties(_i)
    GPUS.append({"id": _i, "name": _p.name,
                 "vram_gb": round(_p.total_memory / 2**30, 1),
                 "cc": f"{_p.major}.{_p.minor}"})

# Replica plan. A T4 has 16 GB: two FP16 Chatterbox replicas often fit, but the
# loader below is deliberately sequential and keeps the last safe count on OOM.
# Set this to 1 if your particular Kaggle image is memory constrained.
REPLICAS_PER_GPU = 2
BASE_DEVICES = [f"cuda:{i}" for i in range(N_GPU)] or ["cpu"]
DEVICES = BASE_DEVICES  # compatibility name; actual worker slots are made in Step 4

# CPU pools: Kaggle's GPU tier gives ~4 vCPUs / 29 GB RAM. That is plenty for
# pipelined ffmpeg encoding + voice-upload transcoding; oversubscribing it
# with dozens of threads would only add churn, so we cap hard.
N_CPU = max(2, min(8, mp.cpu_count() or 4))

if GPUS:
    for g in GPUS:
        print(f"  GPU {g['id']}: {g['name']} · {g['vram_gb']} GB · CC {g['cc']}")
    d = "s" if N_GPU > 1 else ""
    print(f"{N_GPU} GPU{d} → requesting up to {REPLICAS_PER_GPU * N_GPU} replicas ({REPLICAS_PER_GPU}/GPU), loaded one at a time")
else:
    print("!! No CUDA GPU found — running on CPU (consider switching to GPU T4 x2)")
print(f"CPU workers available: {N_CPU}")
print(f"HF cache: {os.environ['HF_HOME']}")


### 3️⃣ Language catalogue · instant

The 23 languages of Chatterbox Multilingual (from `SUPPORTED_LANGUAGES`) with display names/flags for the picker sheet.


In [ ]:
# ── STEP 3 · Language catalogue (23 languages) ───────────────────────────

# code -> (English name, native name, flag)
# Order = picker order: English first, then the top-traffic languages, rest alphabetical.
LANGUAGES = {
    "en": ("English",    "English",   "🇬🇧"),
    "es": ("Spanish",    "Español",   "🇪🇸"),
    "fr": ("French",     "Français",  "🇫🇷"),
    "de": ("German",     "Deutsch",   "🇩🇪"),
    "pt": ("Portuguese", "Português", "🇵🇹"),
    "hi": ("Hindi",      "हिन्दी",     "🇮🇳"),
    "zh": ("Chinese",    "中文",       "🇨🇳"),
    "ja": ("Japanese",   "日本語",     "🇯🇵"),
    "ko": ("Korean",     "한국어",     "🇰🇷"),
    "ar": ("Arabic",     "العربية",   "🇸🇦"),
    "it": ("Italian",    "Italiano",  "🇮🇹"),
    "ru": ("Russian",    "Русский",   "🇷🇺"),
    "tr": ("Turkish",    "Türkçe",    "🇹🇷"),
    "nl": ("Dutch",      "Nederlands","🇳🇱"),
    "pl": ("Polish",     "Polski",    "🇵🇱"),
    "sv": ("Swedish",    "Svenska",   "🇸🇪"),
    "da": ("Danish",     "Dansk",     "🇩🇰"),
    "fi": ("Finnish",    "Suomi",     "🇫🇮"),
    "no": ("Norwegian",  "Norsk",     "🇳🇴"),
    "el": ("Greek",      "Ελληνικά",  "🇬🇷"),
    "he": ("Hebrew",     "עברית",     "🇮🇱"),
    "ms": ("Malay",      "Bahasa Melayu", "🇲🇾"),
    "sw": ("Swahili",    "Kiswahili", "🇰🇪"),
}

DEFAULT_LANG = "en"

def lang_meta(code: str) -> dict:
    name, native, flag = LANGUAGES[code]
    return {"id": code, "name": name, "native": native, "flag": flag}

LANGUAGE_LIST = [lang_meta(c) for c in LANGUAGES]
assert len(LANGUAGE_LIST) == 23, "Chatterbox Multilingual ships exactly 23 languages"
print(f"{len(LANGUAGE_LIST)} languages loaded: {' '.join(LANGUAGES.keys())}")


### 4️⃣ Load Chatterbox V3 — up to four replicas, safely · ~4–8 min first run
Downloads once, then loads one replica at a time. The default asks for **two replicas per T4** (four workers total); if a second replica does not fit, it catches OOM and keeps the first usable. Re-run this cell to retry after a failed download/load.


In [ ]:
# ── STEP 4 · Load Chatterbox V3 — one replica per GPU + fp16 self-test ───

import os, sys, threading, time, types

# --- Perth watermark shim: it's only a provenance tagger; never block TTS on it.
try:
    import perth  # noqa: F401
    HAS_WATERMARK = True
except Exception as e:
    HAS_WATERMARK = False
    _p = types.ModuleType("perth")
    class _DummyWatermarker:
        def apply_watermark(self, wav, sample_rate): return wav
    _p.PerthImplicitWatermarker = _DummyWatermarker
    sys.modules["perth"] = _p
    print(f"(resemble-perth unavailable: {e!s} — outputs will not be watermarked)")

import torch
import numpy as np
from huggingface_hub import snapshot_download
from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES as _CB_LANGS

# Cross-check our hardcoded picker (step 3) against the library we actually
# installed — a future chatterbox could add a language we'd never show.
_lang_diff = set(_CB_LANGS) ^ set(LANGUAGES)
if _lang_diff:
    print(f"!! language list drifted from upstream: {sorted(_lang_diff)} — the UI will show what the MODEL supports")
    for _c in set(_CB_LANGS) - set(LANGUAGES):
        LANGUAGES[_c] = (_CB_LANGS[_c], _CB_LANGS[_c], "🌐")
        LANGUAGE_LIST.append({"id": _c, "name": _CB_LANGS[_c], "native": _CB_LANGS[_c], "flag": "🌐"})

REPO_ID   = "ResembleAI/chatterbox"
T3_MODELS = ["v3", "v2"]          # try newest first
NEEDED    = ["ve.pt", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json",
             "conds.pt", "Cangjie5_TC.json"]

# ---------------------------------------------------------------------------
# OPTIMISATION #1 — weights download ONCE into the persistent cache,
# then every replica loads from disk (parallel CPU->GPU threads after this).
# ---------------------------------------------------------------------------
def _snapshot(t3_file: str) -> str:
    last = None
    for attempt in range(3):
        try:
            return snapshot_download(
                repo_id=REPO_ID, repo_type="model", revision="main",
                allow_patterns=NEEDED + [t3_file],
                token=os.getenv("HF_TOKEN"),
            )
        except Exception as e:
            last = e; print(f"  download attempt {attempt+1} failed: {e!s}"); time.sleep(3)
    raise last

CKPT_DIR, ACTIVE_T3 = None, None
for _t in T3_MODELS:
    try:
        t0 = time.time()
        _file = {"v3": "t3_mtl23ls_v3.safetensors", "v2": "t3_mtl23ls_v2.safetensors"}[_t]
        CKPT_DIR = _snapshot(_file)
        ACTIVE_T3 = _t
        print(f"checkpoint '{_t}' ready in {time.time()-t0:.1f}s -> {CKPT_DIR}")
        break
    except Exception as e:
        print(f"  t3_model={_t} unavailable: {e!s}")
if CKPT_DIR is None:
    raise RuntimeError("could not download any Chatterbox multilingual checkpoint")

# ---------------------------------------------------------------------------
# Replica slots are loaded serially (never four simultaneous CPU->GPU copies).
# Each successful slot is an independent model and can render one text chunk.
# A CUDA OOM is contained: that GPU stops adding slots and already-loaded slots
# remain usable. Re-run this cell after freeing memory to retry downloads/load.
# ---------------------------------------------------------------------------
MODELS: dict[str, ChatterboxMultilingualTTS] = {}
FP16: dict[str, bool] = {}
MODEL_DEVICE: dict[str, str] = {}
WORKERS: list[str] = []

def _load_replica(slot: str, device: str):
    t0 = time.time()
    torch.cuda.empty_cache() if device.startswith("cuda") else None
    m = ChatterboxMultilingualTTS.from_local(CKPT_DIR, device, t3_model=ACTIVE_T3)
    m._cb_lock = threading.Lock()
    m._cb_voice_path = None
    m._cb_default_conds = m.conds
    MODELS[slot], MODEL_DEVICE[slot] = m, device
    return time.time() - t0

for base_device in BASE_DEVICES:
    for replica_no in range(REPLICAS_PER_GPU):
        slot = f"{base_device}#{replica_no}"
        try:
            dt = _load_replica(slot, base_device)
            WORKERS.append(slot)
            print(f"  {slot}: ready in {dt:.1f}s")
        except torch.cuda.OutOfMemoryError:
            print(f"  {slot}: CUDA OOM — keeping {replica_no} safe replica(s) on {base_device}")
            torch.cuda.empty_cache()
            break
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"  {slot}: CUDA OOM — keeping {replica_no} safe replica(s) on {base_device}")
                torch.cuda.empty_cache(); break
            raise
if not WORKERS:
    raise RuntimeError("no Chatterbox model replica could be loaded")
print(f"{len(WORKERS)} replica worker(s) ready (requested up to {len(BASE_DEVICES) * REPLICAS_PER_GPU})")

MODEL_SR = next(iter(MODELS.values())).sr
print(f"sample rate: {MODEL_SR} Hz")

# ---------------------------------------------------------------------------
# OPTIMISATION #3 — per-replica FP16 validation with automatic FP32 fallback.
# T4 tensor cores love fp16, but rather than trusting it blindly we actually
# render a sentence and verify. The winning dtype is applied per-chunk by the
# synthesis engine (step 5) via torch.autocast.
# ---------------------------------------------------------------------------
def _validate_replica(slot: str) -> bool:
    device = MODEL_DEVICE[slot]
    m = MODELS[slot]
    if device == "cpu":
        FP16[slot] = False
        return True
    torch.cuda.set_device(int(device.split(":")[1]))   # smoke test on ITS gpu
    for use_fp16 in (True, False):
        try:
            with m._cb_lock, torch.inference_mode(), \
                 torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                torch.manual_seed(1234)
                wav = m.generate("Chatterbox speaks on every GPU you give it.",
                                 language_id="en")
            w = np.asarray(wav.squeeze().float().cpu(), dtype=np.float32)
            ok = w.size > 0 and np.isfinite(w).all() and float(np.std(w)) > 1e-5
            FP16[slot] = bool(use_fp16 and ok)
            if ok:
                return True
        except Exception as e:
            print(f"  {device}: fp{16 if use_fp16 else 32} smoke test failed: {e!s}")
    FP16[slot] = False
    return False

for slot in WORKERS:
    dev = MODEL_DEVICE[slot]
    ok = _validate_replica(slot)
    tag = "FP16" if FP16.get(slot) else "FP32"
    if dev.startswith("cuda"):
        vram = torch.cuda.memory_allocated(int(dev.split(":")[1])) / 2**30
        print(f"  {dev}: {tag} {'OK' if ok else 'FAILED'} · {vram:.1f} GB VRAM used")
    else:
        print(f"  {dev}: {tag} {'OK' if ok else 'FAILED'}")
print(f"precision plan: { {d: ('fp16' if FP16[d] else 'fp32') for d in WORKERS} }")


### 5️⃣ Parallel synthesis engine · ~5 s (self-test)
Sentence-aware chunks enter a shared queue. Each independent model worker picks the next part, so up to four chunks can synthesize concurrently while final audio stays in chapter order.


In [ ]:
# ── STEP 5 · Parallel synthesis engine (work-stealing across GPUs) ──────

import re, io, time, threading, queue as pyq
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import soundfile as sf

SAMPLE_RATE = MODEL_SR

# Sentence boundaries incl. CJK/Arabic punctuation; blank lines always split.
_SENT_END = re.compile(r"(?<=[\.\!\?。！？!?；;…\u060C\u061F])\s+|\n+")

def split_script(text: str, max_chars: int = 260) -> list[str]:
    text = (text or "").strip()
    if not text: return []
    out = []
    for s in (x.strip() for x in _SENT_END.split(text)):
        if not s: continue
        if len(s) <= max_chars:
            out.append(s); continue
        # long sentence: split on clause punctuation, then on whitespace
        buf = ""
        for piece in re.split(r"(?<=[,;:，；：、])\s*|\s+", s):
            if not piece: continue
            if len(buf) + len(piece) + 1 > max_chars and buf:
                out.append(buf.strip()); buf = piece
            else:
                buf = f"{buf} {piece}".strip()
        if buf: out.append(buf.strip())
    # absolute hard cap — T3 quality degrades well past this anyway
    hard = []
    for c in out:
        while len(c) > max_chars * 2:
            hard.append(c[: max_chars * 2]); c = c[max_chars * 2:]
        hard.append(c)
    return [c for c in hard if c]

def _edge_fade(w: np.ndarray, sr: int, ms: int = 5) -> np.ndarray:
    """5ms in/out fade kills boundary clicks when chunks abut."""
    n = int(sr * ms / 1000)
    if n <= 0 or len(w) <= 2 * n: return w
    w = w.astype(np.float32, copy=True)
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    w[:n] *= ramp
    w[-n:] *= ramp[::-1]
    return w

# ---------------------------------------------------------------------------
# Voice conditionals cache — per replica, per reference clip.
# prepare_conditionals() builds s3-token prompt + xvector + speaker emb; the
# per-call exaggeration delta is patched inside generate() by upstream, so we
# only pay the expensive embedding pass when the voice actually changes.
# ---------------------------------------------------------------------------
def ensure_conditionals(model, voice_ref: str | None, exaggeration: float):
    if voice_ref:                      # cloned voice
        if getattr(model, "_cb_voice_path", None) != voice_ref:
            model.prepare_conditionals(voice_ref, exaggeration=exaggeration)
            model._cb_voice_path = voice_ref
    else:                              # built-in conds.pt voice
        if getattr(model, "_cb_voice_path", None) is not None:
            model.conds = model._cb_default_conds
            model._cb_voice_path = None

# ---------------------------------------------------------------------------
# Work-stealing synthesis: one worker thread per GPU pulls from a shared queue.
# ---------------------------------------------------------------------------
def synth(text: str, language_id: str, voice_ref: str | None, params: dict,
          devices=None, on_progress=None, is_cancelled=None):
    devices = devices or WORKERS
    is_cancelled = is_cancelled or (lambda: False)
    chunks = split_script(text, params.get("chunk_chars", 260))
    total = len(chunks)
    if total == 0:
        return SAMPLE_RATE, np.zeros(1, dtype=np.float32), 0

    work: pyq.Queue = pyq.Queue()
    for i, c in enumerate(chunks):
        work.put((i, c, 0))                       # (index, text, attempts)

    results: dict[int, np.ndarray] = {}
    done = [0]
    errors: list[str] = []
    lock = threading.Lock()

    def worker(slot: str):
        dev = MODEL_DEVICE[slot]
        if dev.startswith("cuda"):
            # pin this thread's CUDA context so no op can fall back to cuda:0
            torch.cuda.set_device(int(dev.split(":")[1]))
        model = MODELS[slot]
        use_fp16 = FP16.get(slot, False) and dev.startswith("cuda")
        while True:
            if is_cancelled():
                return
            try:
                i, chunk, attempts = work.get_nowait()
            except pyq.Empty:
                return
            try:
                seed = int(params.get("seed", 0))
                if seed:                               # deterministic per chunk
                    torch.manual_seed(seed + i)
                    if torch.cuda.is_available():
                        torch.cuda.manual_seed_all(seed + i)
                with model._cb_lock:                   # one generate per GPU
                    ensure_conditionals(model, voice_ref, params["exaggeration"])
                    with torch.inference_mode(), \
                         torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                        wav = model.generate(
                            chunk,
                            language_id=language_id,
                            audio_prompt_path=None,    # conds already primed
                            exaggeration=params["exaggeration"],
                            cfg_weight=params["cfg_weight"],
                            temperature=params["temperature"],
                            repetition_penalty=params["repetition_penalty"],
                            min_p=params["min_p"],
                            top_p=params["top_p"],
                        )
                w = np.asarray(wav.squeeze().detach().float().cpu(), dtype=np.float32)
                if w.size == 0 or not np.isfinite(w).all():
                    raise RuntimeError("model returned bad audio")
                with lock:
                    results[i] = w
                    done[0] += 1
                    n_done = done[0]
                if on_progress:
                    on_progress(n_done, total)
            except Exception as e:
                if is_cancelled():
                    return
                if attempts < 1:                       # retry once, likely on the other GPU
                    work.put((i, chunk, attempts + 1))
                else:
                    with lock:
                        errors.append(f"chunk {i}: {e!s}")
                    return

    threads = [threading.Thread(target=worker, args=(d,), daemon=True,
                                name=f"cb-{d}") for d in devices]
    for t in threads: t.start()
    while any(t.is_alive() for t in threads):
        for t in threads: t.join(timeout=0.2)
        if is_cancelled():                             # workers drop out at next chunk
            for t in threads: t.join()
            raise RuntimeError("cancelled")
    if errors:
        raise RuntimeError(errors[0])

    gap = np.zeros(int(SAMPLE_RATE * params.get("gap_ms", 120) / 1000), dtype=np.float32)
    parts: list[np.ndarray] = []
    for i in range(total):
        parts.append(_edge_fade(results.get(i, np.zeros(1, dtype=np.float32)), SAMPLE_RATE))
        if i != total - 1:
            parts.append(gap)
    return SAMPLE_RATE, np.concatenate(parts), total

# ---------------------------------------------------------------------------
# Encoding on the CPU pool, concurrent with whatever the GPUs are doing next.
# ---------------------------------------------------------------------------
_old = globals().get("_enc_pool")
if _old is not None:                     # notebook re-run: retire the old pool
    try: _old.shutdown(wait=False, cancel_futures=True)
    except Exception: pass
_enc_pool = ThreadPoolExecutor(max_workers=max(2, N_CPU // 2), thread_name_prefix="enc")

def encode(sr: int, wav: np.ndarray, fmt: str) -> tuple[bytes, str]:
    fmt = (fmt or "wav").lower()
    buf = io.BytesIO()
    if fmt == "mp3":
        from pydub import AudioSegment
        pcm = (np.clip(wav, -1, 1) * 32767).astype(np.int16).tobytes()
        AudioSegment(pcm, frame_rate=sr, sample_width=2, channels=1) \
            .export(buf, format="mp3", bitrate="160k", parameters=["-threads", str(N_CPU)])
        return buf.getvalue(), "audio/mpeg"
    if fmt == "opus":
        sf.write(buf, wav, sr, format="OGG", subtype="OPUS"); return buf.getvalue(), "audio/ogg"
    if fmt == "flac":
        sf.write(buf, wav, sr, format="FLAC"); return buf.getvalue(), "audio/flac"
    sf.write(buf, wav, sr, format="WAV", subtype="PCM_16"); return buf.getvalue(), "audio/wav"

def encode_async(sr: int, wav: np.ndarray, fmt: str):
    return _enc_pool.submit(encode, sr, wav, fmt)

# ---- sanity check: both GPUs should appear in the chunk mix ----------------
t0 = time.time()
sr, w, n = synth("Chatterbox speaks fast on two GPUs. Parallel chunks keep both busy. "
                 "This is a self test, so short. And one more sentence for the queue.",
                 "en", None,
                 dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8,
                      repetition_penalty=1.2, min_p=0.05, top_p=1.0,
                      seed=0, gap_ms=120, chunk_chars=260))
print(f"synth ok: {n} chunks -> {len(w)/sr:.2f}s audio in {time.time()-t0:.2f}s wall "
      f"on {len(WORKERS)} model worker(s)")


### 5½️⃣ EPUB + Google Drive resilience
Optional but recommended. Configure Drive once using a Kaggle secret containing a **service-account JSON** and share the target Drive folder with that service account email. This is safer and more reliable in Kaggle than browser OAuth. Completed audio is always persisted locally first.


In [ ]:
# ── STEP 5½ · EPUB helpers + durable output/Google Drive mirror ───────────
# Kaggle Add-ons → Secrets: GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON and
# GOOGLE_DRIVE_FOLDER_ID. Share that Drive folder with the service account.
import os, re, json, threading
from pathlib import Path
from ebooklib import epub
from bs4 import BeautifulSoup

OUTPUT_DIR = Path("/kaggle/working/chatterbox_outputs"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EPUB_DIR = Path("/kaggle/working/chatterbox_epubs"); EPUB_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FOLDER_ID = os.getenv("GOOGLE_DRIVE_FOLDER_ID", "").strip()
DRIVE_SERVICE = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    _secret = _secrets.get_secret("GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON")
    if not DRIVE_FOLDER_ID:
        try: DRIVE_FOLDER_ID = _secrets.get_secret("GOOGLE_DRIVE_FOLDER_ID").strip()
        except Exception: pass
    if _secret and DRIVE_FOLDER_ID:
        from google.oauth2 import service_account
        from googleapiclient.discovery import build
        _info = json.loads(_secret)
        _creds = service_account.Credentials.from_service_account_info(_info, scopes=["https://www.googleapis.com/auth/drive.file"])
        DRIVE_SERVICE = build("drive", "v3", credentials=_creds, cache_discovery=False)
        print("Google Drive mirror enabled")
    else: print("Google Drive mirror off (secrets/folder ID not configured)")
except Exception as e: print(f"Google Drive mirror off: {e!s}")

def safe_filename(title, fallback="chapter"):
    title = re.sub(r'[\\/:*?"<>|\x00-\x1f]+', " ", title or "")
    title = re.sub(r"\s+", " ", title).strip(". ")
    return (title or fallback)[:140]

def mirror_to_drive(path: Path):
    if not DRIVE_SERVICE: return None
    from googleapiclient.http import MediaFileUpload
    media = MediaFileUpload(str(path), resumable=True)
    return DRIVE_SERVICE.files().create(body={"name": path.name, "parents": [DRIVE_FOLDER_ID]}, media_body=media, fields="id").execute()["id"]

def epub_chapters(path: Path):
    book = epub.read_epub(str(path)); chapters=[]
    for item in book.get_items_of_type(epub.ITEM_DOCUMENT):
        soup=BeautifulSoup(item.get_content(), "html.parser")
        for x in soup(["script", "style"]): x.decompose()
        text=soup.get_text(" ", strip=True)
        if len(text) < 40: continue
        heading=soup.find(["h1","h2","h3","title"])
        title=heading.get_text(" ", strip=True) if heading else Path(item.get_name()).stem
        chapters.append({"id": item.get_id(), "title": title or f"Chapter {len(chapters)+1}", "text": text})
    return chapters
print(f"Durable local output: {OUTPUT_DIR}")


### 6️⃣ Build the mobile app UI · instant
The app supports regular scripts, voice clones, and the EPUB panel added below.


In [ ]:
# ── STEP 6 · Build the mobile app UI (inlined single-file frontend) ──────

INDEX_HTML = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover, maximum-scale=5">
<meta name="theme-color" content="#0d0b14">
<meta name="apple-mobile-web-app-capable" content="yes">
<meta name="apple-mobile-web-app-status-bar-style" content="black-translucent">
<meta name="apple-mobile-web-app-title" content="Chatterbox">
<meta name="mobile-web-app-capable" content="yes">
<link rel="manifest" href="/manifest.webmanifest">
<link rel="icon" href="/icon.svg" type="image/svg+xml">
<link rel="apple-touch-icon" href="/icon.svg">
<title>Chatterbox TTS · GPU</title>
<style>
:root{
  --bg:#0d0b14; --surface:#17141f; --surface-2:#1f1b2b; --line:#2b2540;
  --text:#eeeaf6; --muted:#a49eb8; --accent:#a78bfa; --accent-2:#8b5cf6;
  --good:#4ade80; --warn:#f59e0b; --bad:#ef4444;
  --r:14px; --r-sm:10px; --tap:48px;
  --sat:env(safe-area-inset-top); --sab:env(safe-area-inset-bottom);
  --sal:env(safe-area-inset-left); --sar:env(safe-area-inset-right);
}
@media(prefers-color-scheme: light){
  :root{ --bg:#f6f5fb; --surface:#ffffff; --surface-2:#efecf8; --line:#e2ddf2;
         --text:#171226; --muted:#655e7d; --accent:#7c5bf0; --accent-2:#6941e0; }
}
*{ box-sizing:border-box; -webkit-tap-highlight-color:transparent; }
html,body{ margin:0; padding:0; background:var(--bg); color:var(--text);
  font: 16px/1.45 -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue",
        Arial, "Noto Sans", sans-serif; -webkit-font-smoothing:antialiased;
  overscroll-behavior-y:none; }
body{ min-height:100dvh; padding-left:var(--sal); padding-right:var(--sar);
  padding-bottom: calc(var(--sab) + 92px); }
button, input, textarea, select{ font:inherit; color:inherit; }
button{ background:none; border:0; cursor:pointer; }

/* header */
header{ position:sticky; top:0; z-index:5; background:color-mix(in oklab, var(--bg) 88%, transparent);
  backdrop-filter:saturate(140%) blur(10px); -webkit-backdrop-filter:saturate(140%) blur(10px);
  padding: calc(var(--sat) + 10px) 16px 10px; border-bottom:1px solid var(--line);
  display:flex; align-items:center; gap:10px; }
.logo{ width:32px; height:32px; border-radius:9px; background:
  linear-gradient(135deg, var(--accent), var(--accent-2)); display:grid; place-items:center;
  color:#fff; font-weight:800; font-size:14px; }
.title{ font-weight:700; font-size:16px; letter-spacing:.2px; }
.sub{ font-size:11px; color:var(--muted); margin-top:1px; }
.pill{ margin-left:auto; display:inline-flex; align-items:center; gap:6px;
  padding:6px 10px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:999px; font-size:12px; color:var(--muted); }
.dot{ width:8px; height:8px; border-radius:50%; background:var(--good); box-shadow:0 0 0 3px color-mix(in oklab, var(--good) 30%, transparent);}

/* main */
main{ max-width: 680px; margin: 0 auto; padding: 12px 14px; display:flex; flex-direction:column; gap:12px; }
.card{ background:var(--surface); border:1px solid var(--line); border-radius:var(--r); padding:14px; }
.label{ font-size:12px; color:var(--muted); text-transform:uppercase; letter-spacing:.6px; margin-bottom:8px; display:flex; justify-content:space-between; align-items:center; }

/* picker cards (language / voice) */
.pickgrid{ display:grid; grid-template-columns: 1fr 1fr; gap:10px; }
.pick{ display:flex; align-items:center; gap:12px; padding:12px; min-height:var(--tap); text-align:left; }
.pick .avatar{ width:44px; height:44px; border-radius:50%; flex:0 0 auto;
  background:linear-gradient(135deg, var(--accent), var(--accent-2));
  color:#fff; display:grid; place-items:center; font-weight:700; font-size:16px; }
.pick .who{ flex:1; min-width:0; }
.pick .who b{ display:block; font-size:15px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
.pick .who span{ font-size:11.5px; color:var(--muted); display:block; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
.pick .chev{ color:var(--muted); font-size:20px; }
.pick .k{ font-size:10px; text-transform:uppercase; letter-spacing:.6px; color:var(--muted); margin-bottom:2px; }
@media (max-width:420px){ .pickgrid{ grid-template-columns:1fr; } }

/* script */
textarea{ width:100%; min-height:40vh; max-height:62vh; background:var(--surface);
  border:1px solid var(--line); border-radius:var(--r); padding:14px; color:var(--text);
  font-size:16px; line-height:1.5; resize:vertical; }
textarea:focus{ outline:2px solid var(--accent); outline-offset:1px; }
.meter{ display:flex; justify-content:space-between; font-size:12px; color:var(--muted); padding:6px 4px 0; }

/* controls */
.row{ display:grid; grid-template-columns: 1fr auto; gap:10px; align-items:center; margin-bottom:12px; }
.row:last-child{ margin-bottom:0; }
.row .name{ font-size:14px; }
.row .hint{ display:block; font-size:11px; color:var(--muted); margin-top:1px; }
.val{ font-variant-numeric: tabular-nums; color:var(--muted); font-size:13px; min-width:56px; text-align:right; }
input[type=range]{ -webkit-appearance:none; appearance:none; width:100%; height:36px; background:transparent; }
input[type=range]::-webkit-slider-runnable-track{ height:6px; border-radius:3px; background:var(--surface-2); }
input[type=range]::-moz-range-track{ height:6px; border-radius:3px; background:var(--surface-2); }
input[type=range]::-webkit-slider-thumb{ -webkit-appearance:none; appearance:none; width:26px; height:26px; border-radius:50%;
  background:var(--accent); margin-top:-10px; border:3px solid var(--bg); box-shadow:0 2px 6px rgba(0,0,0,.3);}
input[type=range]::-moz-range-thumb{ width:22px; height:22px; border-radius:50%; background:var(--accent); border:3px solid var(--bg); }
input[type=number]{ width:100%; min-height:44px; padding:0 12px; background:var(--surface-2); border:1px solid var(--line); border-radius:10px; }

details.adv summary{ list-style:none; display:flex; align-items:center; gap:8px; min-height:var(--tap);
  color:var(--muted); font-size:14px; font-weight:600; cursor:pointer; }
details.adv summary::-webkit-details-marker{ display:none; }
details.adv summary .chev{ transition:transform .15s ease; font-size:18px; }
details.adv[open] summary .chev{ transform:rotate(90deg); }
details.adv .body{ padding-top:4px; }

.segmented{ display:grid; grid-auto-flow:column; grid-auto-columns:1fr; gap:6px; background:var(--surface-2); padding:4px;
  border-radius:12px; border:1px solid var(--line); }
.segmented button{ min-height:40px; border-radius:9px; font-weight:600; font-size:13px; color:var(--muted); }
.segmented button.on{ background:var(--surface); color:var(--text); box-shadow:0 1px 2px rgba(0,0,0,.15); }

/* sticky action */
.fab-wrap{ position:fixed; left:0; right:0; bottom:0; padding: 10px 14px calc(var(--sab) + 10px);
  background: linear-gradient(to top, var(--bg) 55%, transparent);
  z-index:4; }
.fab-wrap .inner{ max-width:680px; margin:0 auto; display:flex; gap:10px; align-items:center; }
.fab{ flex:1; min-height:56px; background:var(--accent); color:#fff; border-radius:14px;
  font-weight:700; font-size:16px; letter-spacing:.2px; display:inline-flex; align-items:center; justify-content:center; gap:10px;
  box-shadow: 0 6px 20px rgba(124,91,240,.35); transition: transform .06s ease; }
.fab:active{ transform: scale(.98); }
.fab[disabled]{ background:var(--surface-2); color:var(--muted); box-shadow:none; }
.fab .spinner{ width:18px; height:18px; border-radius:50%; border:2px solid rgba(255,255,255,.4); border-top-color:#fff; animation:spin .8s linear infinite; }
@keyframes spin{ to{ transform:rotate(360deg); } }
.icon-btn{ width:56px; height:56px; border-radius:14px; background:var(--surface); border:1px solid var(--line); display:grid; place-items:center; color:var(--text); font-size:18px; }

/* progress bar */
.progress{ position:fixed; top:0; left:0; right:0; height:3px; background:transparent; z-index:10; pointer-events:none; }
.progress .bar{ height:100%; width:0%; background:linear-gradient(90deg, var(--accent), var(--accent-2)); transition: width .2s ease; }

/* audio result */
.result{ display:none; }
.result.on{ display:block; }
.result audio{ width:100%; margin-top:8px; }
.result .stats{ display:flex; gap:12px; font-size:12px; color:var(--muted); flex-wrap:wrap; margin-top:8px; }
.result .stats b{ color:var(--text); font-weight:600; }
.result .dlrow{ display:flex; gap:8px; margin-top:10px; }
.btn{ flex:1; min-height:44px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:10px; font-weight:600; font-size:14px; color:var(--text); display:inline-flex; align-items:center; justify-content:center; gap:8px; text-decoration:none; }
.btn.primary{ background:var(--accent); color:#fff; border-color:transparent; }

/* history */
.hist-item{ display:flex; align-items:center; gap:10px; padding:10px; border-radius:10px; }
.hist-item + .hist-item{ border-top:1px solid var(--line); border-radius:0; }
.hist-item .who{ flex:1; min-width:0; }
.hist-item .who b{ display:block; font-size:14px; }
.hist-item .who span{ font-size:12px; color:var(--muted); white-space:nowrap; overflow:hidden; text-overflow:ellipsis; display:block; }
.hist-item .play{ width:40px; height:40px; border-radius:50%; background:var(--accent); color:#fff; display:grid; place-items:center; }

/* bottom sheet */
.sheet-back{ position:fixed; inset:0; background:rgba(0,0,0,.55); opacity:0; pointer-events:none; transition:opacity .18s ease; z-index:20; }
.sheet-back.on{ opacity:1; pointer-events:auto; }
.sheet{ position:fixed; left:0; right:0; bottom:0; z-index:21; background:var(--surface); border-top-left-radius:20px; border-top-right-radius:20px;
  transform: translateY(100%); transition: transform .22s cubic-bezier(.2,.8,.2,1);
  max-height: 88dvh; display:flex; flex-direction:column; padding-bottom: var(--sab); }
.sheet.on{ transform: translateY(0); }
.sheet .grabber{ width:44px; height:5px; background:var(--line); border-radius:3px; margin: 8px auto 4px; }
.sheet .head{ display:flex; align-items:center; padding: 6px 14px 10px; gap:10px; border-bottom:1px solid var(--line); }
.sheet .head b{ font-size:16px; }
.sheet .head button{ margin-left:auto; color:var(--muted); font-size:15px; min-height:44px; padding: 0 10px; }
.sheet .search{ padding: 10px 14px; border-bottom:1px solid var(--line); }
.sheet .search input{ width:100%; min-height:44px; padding: 0 12px; background:var(--surface-2); border:1px solid var(--line); border-radius:10px; }
.sheet .list{ overflow-y:auto; padding: 6px 8px 8px; }
.sheet .group{ font-size:11px; color:var(--muted); text-transform:uppercase; letter-spacing:.7px; padding: 12px 8px 6px; }
.sheet .item{ display:flex; align-items:center; gap:12px; padding:12px 10px; border-radius:12px; min-height:var(--tap); width:100%; text-align:left; }
.sheet .item:active{ background: var(--surface-2); }
.sheet .item.on{ background: color-mix(in oklab, var(--accent) 15%, transparent); }
.sheet .item .avatar{ width:36px; height:36px; border-radius:50%; background:linear-gradient(135deg, var(--accent), var(--accent-2)); color:#fff; display:grid; place-items:center; font-weight:700; font-size:14px; }
.sheet .item .meta b{ display:block; font-size:15px; }
.sheet .item .meta span{ font-size:12px; color:var(--muted); }
.sheet .item .check{ margin-left:auto; color:var(--accent); opacity:0; }
.sheet .item.on .check{ opacity:1; }
.sheet .item .del{ margin-left:auto; color:var(--muted); font-size:18px; padding:6px 10px; }
.sheet .upload{ display:flex; align-items:center; justify-content:center; gap:10px; margin:10px 8px;
  min-height:var(--tap); border:1.5px dashed var(--line); border-radius:12px; color:var(--accent);
  font-weight:600; font-size:14px; width:calc(100% - 16px); }
.sheet .upload .spinner{ width:16px; height:16px; border-radius:50%; border:2px solid color-mix(in oklab, var(--accent) 40%, transparent); border-top-color:var(--accent); animation:spin .8s linear infinite; }
.sheet .note{ font-size:11.5px; color:var(--muted); padding:4px 14px 0; }

/* toast */
.toast{ position:fixed; left:14px; right:14px; bottom: calc(var(--sab) + 110px); z-index:30;
  background:var(--bad); color:#fff; padding:12px 14px; border-radius:12px; box-shadow: 0 8px 30px rgba(0,0,0,.3);
  transform: translateY(20px); opacity:0; transition: all .2s ease; pointer-events:none; text-align:center; }
.toast.on{ transform:none; opacity:1; }

.hide{ display:none !important; }
</style>
</head>
<body>
  <div class="progress"><div class="bar" id="pbar"></div></div>

  <header>
    <div class="logo">C</div>
    <div>
      <div class="title">Chatterbox TTS</div>
      <div class="sub" id="sub">connecting…</div>
    </div>
    <div class="pill"><span class="dot" id="statusDot"></span><span id="statusText">…</span></div>
  </header>

  <main>
    <div class="pickgrid">
      <button class="card pick" id="langBtn" aria-label="Choose language">
        <div>
          <div class="k">Language</div>
          <div class="who"><b id="lName">🇬🇧 English</b><span id="lMeta">English</span></div>
        </div>
        <div class="chev">›</div>
      </button>
      <button class="card pick" id="voiceBtn" aria-label="Choose voice">
        <div class="avatar" id="vAvatar">C</div>
        <div style="min-width:0; flex:1;">
          <div class="k">Voice</div>
          <div class="who"><b id="vName">Default voice</b><span id="vMeta">built-in conds.pt</span></div>
        </div>
        <div class="chev">›</div>
      </button>
    </div>

    <div>
      <textarea id="script" spellcheck="true" autocapitalize="sentences"
        enterkeyhint="enter" inputmode="text"
        placeholder="Paste or type your script here — any of 23 languages…">Chatterbox speaks twenty-three languages, clones any voice from a few seconds of reference audio, and on this Kaggle box it renders across two T4 GPUs in parallel. Try a long paragraph — you will watch both GPUs share the work.</textarea>
      <div class="meter"><span id="charCount">0 chars</span><span id="estAudio">≈ 0s audio</span></div>
    </div>

    <div class="card">
      <div class="label">EPUB → chapter audio</div>
      <div class="hint" style="margin:6px 0 10px">Upload a book, tick the chapters to render. Each finished file is named after its chapter and saved locally (and mirrored to Drive if enabled).</div>
      <input type="file" id="epubFile" accept=".epub,application/epub+zip" style="display:none">
      <button class="btn" id="epubPick">Upload EPUB</button>
      <div id="epubInfo" class="hint" style="margin-top:9px"></div>
      <div id="epubChapters" style="max-height:230px;overflow:auto;margin-top:8px"></div>
      <button class="btn primary" id="epubGo" style="display:none;margin-top:10px">Convert selected chapters</button>
    </div>

    <div class="card">
      <div class="row">
        <div class="name">Exaggeration<span class="hint">0 = flat · 0.5 = neutral · 1+ = theatrical</span></div>
        <div class="val" id="exVal">0.50</div>
      </div>
      <input type="range" id="exaggeration" min="0" max="2" step="0.05" value="0.5" style="margin-bottom:14px">
      <div class="row">
        <div class="name">CFG / pace<span class="hint">lower = faster speech · higher = slower &amp; more guided</span></div>
        <div class="val" id="cfgVal">0.50</div>
      </div>
      <input type="range" id="cfg" min="0" max="1" step="0.05" value="0.5" style="margin-bottom:14px">
      <div class="row">
        <div class="name">Temperature<span class="hint">lower = steadier · higher = more varied</span></div>
        <div class="val" id="tempVal">0.80</div>
      </div>
      <input type="range" id="temperature" min="0.05" max="1.5" step="0.05" value="0.8">
    </div>

    <div class="card">
      <div class="row">
        <div class="name">Seed<span class="hint">0 = random · any other number reproduces the render exactly</span></div>
      </div>
      <input type="number" id="seed" min="0" max="99999999" step="1" value="0" inputmode="numeric">
    </div>

    <details class="card adv" id="advCard">
      <summary><span class="chev">›</span> Advanced <span style="margin-left:auto; font-size:11px; font-weight:400;">defaults match the official V3 space</span></summary>
      <div class="body">
        <div class="row">
          <div class="name">Repetition penalty<span class="hint">raise if it stutters/loops</span></div>
          <div class="val" id="repVal">1.20</div>
        </div>
        <input type="range" id="rep" min="1" max="5" step="0.1" value="1.2" style="margin-bottom:14px">
        <div class="row">
          <div class="name">min_p</div>
          <div class="val" id="minpVal">0.05</div>
        </div>
        <input type="range" id="minp" min="0" max="0.25" step="0.01" value="0.05" style="margin-bottom:14px">
        <div class="row">
          <div class="name">top_p</div>
          <div class="val" id="toppVal">1.00</div>
        </div>
        <input type="range" id="topp" min="0.5" max="1" step="0.01" value="1" style="margin-bottom:14px">
        <div class="row">
          <div class="name">Gap between sentences</div>
          <div class="val" id="gapVal">120 ms</div>
        </div>
        <input type="range" id="gap" min="0" max="500" step="10" value="120" style="margin-bottom:14px">
        <div class="row">
          <div class="name">Chunk size<span class="hint">smaller = better GPU balance · larger = fewer boundary seams</span></div>
          <div class="val" id="chunkVal">260 chars</div>
        </div>
        <input type="range" id="chunk" min="140" max="400" step="20" value="260">
      </div>
    </details>

    <div class="card">
      <div class="label">Format</div>
      <div class="segmented" id="fmt" role="radiogroup">
        <button data-v="wav"  class="on">WAV</button>
        <button data-v="mp3">MP3</button>
        <button data-v="opus">Opus</button>
        <button data-v="flac">FLAC</button>
      </div>
    </div>

    <div class="card result" id="result">
      <div class="label"><span>Latest render</span><span id="resStamp"></span></div>
      <audio id="player" controls preload="metadata" playsinline></audio>
      <div class="stats">
        <span><b id="resDur">0.00s</b> audio</span>
        <span>rendered in <b id="resTime">0.00s</b></span>
        <span>RTF <b id="resRTF">0.00×</b></span>
        <span id="resChunks"></span>
      </div>
      <div class="dlrow">
        <a class="btn primary" id="downloadBtn" download="chatterbox.wav">↓ Download</a>
        <button class="btn" id="shareBtn">Share</button>
      </div>
    </div>

    <div class="card hide" id="histCard">
      <div class="label">History</div>
      <div id="histList"></div>
    </div>
  </main>

  <div class="fab-wrap">
    <div class="inner">
      <button class="icon-btn" id="stopBtn" title="Stop" aria-label="Stop" style="display:none;">■</button>
      <button class="fab" id="goBtn"><span id="goLabel">Generate</span></button>
    </div>
  </div>

  <div class="sheet-back" id="sheetBack"></div>

  <div class="sheet" id="langSheet" role="dialog" aria-label="Choose a language">
    <div class="grabber"></div>
    <div class="head"><b>Language</b><button class="sheetClose">Done</button></div>
    <div class="search"><input id="langSearch" placeholder="Search languages…" enterkeyhint="search"></div>
    <div class="list" id="langList"></div>
  </div>

  <div class="sheet" id="voiceSheet" role="dialog" aria-label="Choose a voice">
    <div class="grabber"></div>
    <div class="head"><b>Voice</b><button class="sheetClose">Done</button></div>
    <div class="list" id="voiceList"></div>
    <button class="upload" id="uploadBtn"><span id="uploadLabel">＋ Clone a voice from an audio clip…</span></button>
    <div class="note">5–15 s of clear speech works best. The clip is transcoded to 24 kHz WAV and used only inside this Kaggle session.</div>
    <input type="file" id="fileInput" accept="audio/*,.wav,.mp3,.m4a,.ogg,.oga,.webm,.flac,.aac" style="display:none">
  </div>

  <div class="toast" id="toast">…</div>

<script>
const $ = s => document.querySelector(s);
const state = { languages: [], lang: 'en', voices: [], voice: 'default',
                fmt: 'wav', busy: false, job: null, ctrl: null, history: [] };

function toast(msg, kind='bad'){
  const t = $('#toast'); t.textContent = msg;
  t.style.background = kind==='good' ? 'var(--good)' : 'var(--bad)';
  t.classList.add('on'); clearTimeout(toast._t);
  toast._t = setTimeout(()=>t.classList.remove('on'), 3400);
}
function esc(s){ return (s||'').replace(/[&<>"']/g, c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c])); }
function initials(v){ const n=(v.name||'').replace(/[^a-z0-9]/gi,''); return (n[0]||'C').toUpperCase(); }

/* ---------- language ---------- */
function setLang(code){
  const l = state.languages.find(x=>x.id===code) || state.languages[0]; if(!l) return;
  state.lang = l.id;
  $('#lName').textContent = `${l.flag} ${l.name}`;
  $('#lMeta').textContent = l.native === l.name ? l.id : l.native;
  try{ localStorage.setItem('cb.lang', l.id); }catch{}
  renderLangSheet(($('#langSearch').value||''));
}
function renderLangSheet(q){
  q = (q||'').trim().toLowerCase();
  const list = $('#langList'); list.innerHTML='';
  let any = false;
  for(const l of state.languages){
    if(q && !(l.id.includes(q) || l.name.toLowerCase().includes(q) || (l.native||'').toLowerCase().includes(q))) continue;
    any = true;
    const b = document.createElement('button'); b.className='item'+(l.id===state.lang?' on':'');
    b.innerHTML = `<span class="avatar">${l.flag}</span>
      <span class="meta"><b>${esc(l.name)}</b><span>${esc(l.native)} · ${l.id}</span></span>
      <span class="check">✓</span>`;
    b.addEventListener('click', ()=>{ setLang(l.id); closeSheets(); });
    list.appendChild(b);
  }
  if(!any) list.innerHTML = '<div style="padding:20px;text-align:center;color:var(--muted)">No matches.</div>';
}

/* ---------- voice ---------- */
function setVoice(id){
  const v = state.voices.find(x=>x.id===id) || state.voices[0]; if(!v) return;
  state.voice = v.id;
  $('#vName').textContent = v.name;
  $('#vMeta').textContent = v.kind === 'builtin' ? 'built-in conds.pt'
                          : `cloned · ${v.seconds ? v.seconds.toFixed(1)+'s ref' : 'your clip'}`;
  $('#vAvatar').textContent = v.kind === 'builtin' ? 'C' : initials(v);
  try{ localStorage.setItem('cb.voice', v.id); }catch{}
  renderVoiceSheet();
}
function renderVoiceSheet(){
  const list = $('#voiceList'); list.innerHTML='';
  const mk = (v)=>{
    const b = document.createElement('button'); b.className='item'+(v.id===state.voice?' on':'');
    b.innerHTML = `<span class="avatar">${v.kind==='builtin'?'C':initials(v)}</span>
      <span class="meta"><b>${esc(v.name)}</b><span>${v.kind==='builtin'?'built-in Chatterbox voice':'cloned voice'+(v.seconds?` · ${v.seconds.toFixed(1)}s`:'')}</span></span>
      ${v.kind==='builtin'
        ? '<span class="check">✓</span>'
        : '<span class="check" style="margin-right:2px">✓</span><span class="del" data-del="1" aria-label="Delete">🗑</span>'}`;
    b.addEventListener('click', e=>{
      if(e.target.dataset.del){ delVoice(v.id); return; }
      setVoice(v.id); closeSheets();
    });
    list.appendChild(b);
  };
  const def = state.voices.find(v=>v.kind==='builtin');
  if(def){ const h=document.createElement('div'); h.className='group'; h.textContent='Built-in'; list.appendChild(h); mk(def); }
  const clones = state.voices.filter(v=>v.kind!=='builtin');
  if(clones.length){ const h=document.createElement('div'); h.className='group'; h.textContent='Your cloned voices'; list.appendChild(h); clones.forEach(mk); }
}
async function delVoice(id){
  try{
    const r = await fetch('/api/voices/'+id, {method:'DELETE'});
    if(!r.ok) throw new Error();
    state.voices = state.voices.filter(v=>v.id!==id);
    if(state.voice===id) setVoice('default'); else renderVoiceSheet();
    toast('Voice deleted','good');
  }catch{ toast('Delete failed'); }
}
async function uploadVoice(file){
  if(!file) return;
  if(file.size > 30*1024*1024){ toast('Clip too large (30 MB max)'); return; }
  const lbl = $('#uploadLabel');
  lbl.innerHTML = '<span class="spinner"></span> Transcoding &amp; uploading…';
  $('#uploadBtn').style.pointerEvents = 'none';
  try{
    const fd = new FormData(); fd.append('file', file, file.name||'clip.webm');
    const r = await fetch('/api/voices', {method:'POST', body:fd});
    const j = await r.json();
    if(!r.ok) throw new Error(j.detail||'upload failed');
    state.voices.push(j.voice);
    setVoice(j.voice.id);
    toast(`Voice “${j.voice.name}” ready`, 'good');
  }catch(e){ toast(e.message||'Upload failed'); }
  finally{
    lbl.textContent = '＋ Clone a voice from an audio clip…';
    $('#uploadBtn').style.pointerEvents = '';
    $('#fileInput').value = '';
  }
}

/* ---------- sheets ---------- */
let openSheetEl = null;
function openSheet(el){ closeSheets(); openSheetEl = el; el.classList.add('on'); $('#sheetBack').classList.add('on'); }
function closeSheets(){ for(const s of document.querySelectorAll('.sheet')) s.classList.remove('on');
  $('#sheetBack').classList.remove('on'); openSheetEl = null; }

/* ---------- format ---------- */
function fmt(v){ state.fmt = v; for(const b of $('#fmt').children) b.classList.toggle('on', b.dataset.v===v);
  $('#downloadBtn').setAttribute('download', 'chatterbox.'+v);
  try{ localStorage.setItem('cb.fmt', v); }catch{} }

/* ---------- meters ---------- */
function updateMeters(){
  const c = $('#script').value.length;
  $('#charCount').textContent = `${c.toLocaleString()} chars`;
  const s = c / 15;   // chatterbox ≈ 15 spoken chars/sec in latin scripts
  $('#estAudio').textContent = `≈ ${s>=60 ? (s/60).toFixed(1)+'m' : s.toFixed(1)+'s'} audio`;
}

/* ---------- history ---------- */
function pushHistory(item){
  state.history.unshift(item); state.history = state.history.slice(0, 6);
  const list = $('#histList'); list.innerHTML='';
  for(const h of state.history){
    const el = document.createElement('div'); el.className='hist-item';
    el.innerHTML = `<button class="play" aria-label="Play">▶</button>
      <div class="who"><b>${esc(h.label)}</b><span>${esc(h.preview)}</span></div>
      <a class="btn" style="flex:0 0 auto; min-width:64px" href="${h.url}" download="chatterbox.${h.fmt}">↓</a>`;
    el.querySelector('.play').addEventListener('click', ()=>{ const p=$('#player'); p.src=h.url; p.play(); });
    list.appendChild(el);
  }
  $('#histCard').classList.toggle('hide', !state.history.length);
}

/* ---------- generate ---------- */
function setBusy(b){ state.busy = b;
  $('#goBtn').disabled = b;
  $('#goLabel').innerHTML = b ? '<span class="spinner"></span> Rendering…' : 'Generate';
  $('#stopBtn').style.display = b ? 'grid' : 'none';
  $('#pbar').style.width = b ? '5%' : '0%';
}

async function loadStatus(){
  try{
    const r = await fetch('/api/status'); const j = await r.json();
    state.languages = j.languages; state.voices = j.voices;
    $('#statusText').textContent = j.n_gpus ? `${j.n_gpus} GPU${j.n_gpus===1?'':'s'}` : 'CPU';
    const gpu = (j.gpus&&j.gpus.length)? j.gpus.map(g=>g.name.replace('Tesla ','')).join(' + ') : 'CPU';
    $('#sub').textContent = `${j.model} · ${gpu}${j.fp16 ? ' · fp16' : ''}`;
    const saved = localStorage.getItem('cb.lang');
    setLang(saved && j.languages.some(l=>l.id===saved) ? saved : 'en');
    const savedV = localStorage.getItem('cb.voice');
    setVoice(savedV && j.voices.some(v=>v.id===savedV) ? savedV : 'default');
    const savedFmt = localStorage.getItem('cb.fmt'); if(savedFmt) fmt(savedFmt);
    renderLangSheet(''); renderVoiceSheet();
  }catch(e){ $('#sub').textContent = 'backend not reachable';
    $('#statusDot').style.background = 'var(--bad)'; $('#statusDot').style.boxShadow = 'none';
    $('#statusText').textContent = 'offline';
    toast('Backend not reachable'); }
}

async function generate(){
  const text = $('#script').value.trim();
  if(!text){ toast('Script is empty'); return; }
  if(state.busy) return;
  setBusy(true);
  const payload = {
    text,
    language_id: state.lang,
    voice: state.voice,
    exaggeration: parseFloat($('#exaggeration').value),
    cfg_weight: parseFloat($('#cfg').value),
    temperature: parseFloat($('#temperature').value),
    seed: parseInt($('#seed').value||'0',10) || 0,
    repetition_penalty: parseFloat($('#rep').value),
    min_p: parseFloat($('#minp').value),
    top_p: parseFloat($('#topp').value),
    gap_ms: parseInt($('#gap').value,10),
    chunk_chars: parseInt($('#chunk').value,10),
    format: state.fmt,
  };

  const ctrl = new AbortController(); state.ctrl = ctrl;
  const t0 = performance.now();
  try{
    const jr = await fetch('/api/jobs', {method:'POST', headers:{'content-type':'application/json'},
                                          body: JSON.stringify(payload), signal: ctrl.signal});
    if(!jr.ok){ const j = await jr.json().catch(()=>({})); throw new Error(j.detail||'server rejected'); }
    const {job_id} = await jr.json(); state.job = job_id;

    const meta = await new Promise((resolve, reject)=>{
      const es = new EventSource('/api/jobs/'+job_id+'/events');
      ctrl.signal.addEventListener('abort', ()=>{ es.close(); reject(new Error('aborted')); });
      es.addEventListener('progress', e=>{
        try{ const d = JSON.parse(e.data);
          $('#pbar').style.width = (5 + 88*(d.done/Math.max(1,d.total))) + '%';
          $('#goLabel').innerHTML = `<span class="spinner"></span> ${d.done}/${d.total} chunks`;
        }catch{}
      });
      es.addEventListener('done', e=>{ es.close(); resolve(JSON.parse(e.data)); });
      es.addEventListener('error', e=>{ es.close();
        let msg = 'stream error';
        try{ msg = JSON.parse(e.data).message || msg; }catch{}
        reject(new Error(msg)); });
    });

    $('#pbar').style.width = '96%';
    const ar = await fetch('/api/jobs/'+job_id+'/audio', {signal: ctrl.signal});
    if(!ar.ok) throw new Error('audio fetch failed');
    const blob = await ar.blob();
    const url  = URL.createObjectURL(blob);
    const m = (meta && Object.keys(meta).length) ? meta : JSON.parse(ar.headers.get('X-Meta') || '{}');

    const dur = m.audio_s || 0, took = (performance.now()-t0)/1000;
    $('#player').src = url;
    $('#resDur').textContent  = dur.toFixed(2)+'s';
    $('#resTime').textContent = (m.wall_s || took).toFixed(2)+'s';
    $('#resRTF').textContent  = (dur>0 ? ((m.wall_s||took)/dur).toFixed(2) : '-')+'×';
    $('#resChunks').textContent = m.chunks ? `${m.chunks} chunk${m.chunks>1?'s':''} · ${m.gpus||''} GPU(s)` : '';
    $('#resStamp').textContent = new Date().toLocaleTimeString();
    const a = $('#downloadBtn'); a.href = url; a.setAttribute('download','chatterbox.'+state.fmt);
    $('#result').classList.add('on');
    const lname = (state.languages.find(l=>l.id===state.lang)||{}).name || state.lang;
    const vname = (state.voices.find(v=>v.id===state.voice)||{}).name || 'voice';
    pushHistory({ url, fmt: state.fmt, label: `${lname} · ${vname}`, preview: text.slice(0,80) });
    $('#pbar').style.width = '100%';
    try{ await $('#player').play(); }catch{}
  }catch(e){
    if(e.name!=='AbortError' && e.message!=='aborted') toast(e.message||'Generation failed');
  }finally{
    setTimeout(()=>{ if(!state.busy) $('#pbar').style.width='0%'; }, 500);
    state.job=null; state.ctrl=null; setBusy(false);
  }
}

async function stopJob(){
  if(state.ctrl) state.ctrl.abort();
  if(state.job){ try{ await fetch('/api/jobs/'+state.job, {method:'DELETE'}); }catch{} }
}

async function share(){
  const p = $('#player'); if(!p.src) return;
  try{
    const blob = await (await fetch(p.src)).blob();
    const file = new File([blob], 'chatterbox.'+state.fmt, {type: blob.type});
    if(navigator.canShare && navigator.canShare({files:[file]})){
      await navigator.share({ files:[file], title:'Chatterbox render' });
    }else{
      $('#downloadBtn').click();
    }
  }catch(e){ /* user cancelled */ }
}

/* ---------- EPUB ---------- */
let epubBook = null;
function chapterPayload(){ return {
  language_id: state.lang, voice: state.voice, format: state.fmt,
  exaggeration:+$('#exaggeration').value, cfg_weight:+$('#cfg').value,
  temperature:+$('#temperature').value, seed:parseInt($('#seed').value||'0',10)||0,
  repetition_penalty:+$('#rep').value, min_p:+$('#minp').value, top_p:+$('#topp').value,
  gap_ms:+$('#gap').value, chunk_chars:+$('#chunk').value
}; }
async function uploadEpub(file){
  if(!file) return; $('#epubInfo').textContent='Reading EPUB…';
  const fd=new FormData(); fd.append('file',file);
  try{
    const r=await fetch('/api/epub',{method:'POST',body:fd}); const j=await r.json();
    if(!r.ok) throw new Error(j.detail||'EPUB upload failed'); epubBook=j;
    $('#epubInfo').textContent=`${j.chapters.length} readable chapters — select what to convert`;
    $('#epubChapters').innerHTML=j.chapters.map((c,i)=>`<label style="display:flex;gap:8px;padding:7px 0;border-bottom:1px solid var(--line);font-size:13px"><input type="checkbox" class="epubCheck" value="${esc(c.id)}" ${i===0?'checked':''}><span><b>${esc(c.title)}</b><br><small>${c.chars.toLocaleString()} chars</small></span></label>`).join('');
    $('#epubGo').style.display='inline-flex';
  }catch(e){ $('#epubInfo').textContent=e.message; toast(e.message); }
}
async function convertEpub(){
  if(!epubBook) return; const chapter_ids=[...document.querySelectorAll('.epubCheck:checked')].map(x=>x.value);
  if(!chapter_ids.length) return toast('Select at least one chapter');
  $('#epubGo').disabled=true; $('#epubGo').textContent='Queueing chapters…';
  try { const r=await fetch('/api/epub/'+epubBook.epub_id+'/jobs',{method:'POST',headers:{'content-type':'application/json'},body:JSON.stringify({...chapterPayload(),chapter_ids})}); const j=await r.json(); if(!r.ok) throw new Error(j.detail||'Could not queue chapters'); $('#epubInfo').textContent=`Queued ${j.jobs.length} chapter(s). They continue on Kaggle even if this page disconnects; files go to ${j.saved_to}${j.drive_enabled?' and Google Drive':''}.`; }
  catch(e){toast(e.message);}
  finally{$('#epubGo').disabled=false; $('#epubGo').textContent='Convert selected chapters';}
}

/* ---------- wire up ---------- */
$('#langBtn').addEventListener('click', ()=>{ renderLangSheet(''); openSheet($('#langSheet'));
  setTimeout(()=>$('#langSearch').focus({preventScroll:true}), 200); });
$('#voiceBtn').addEventListener('click', ()=>{ renderVoiceSheet(); openSheet($('#voiceSheet')); });
$('#sheetBack').addEventListener('click', closeSheets);
for(const b of document.querySelectorAll('.sheetClose')) b.addEventListener('click', closeSheets);
$('#langSearch').addEventListener('input', e=>renderLangSheet(e.target.value));
$('#uploadBtn').addEventListener('click', ()=>$('#fileInput').click());
$('#epubPick').addEventListener('click', ()=>$('#epubFile').click());
$('#epubFile').addEventListener('change', e=>uploadEpub(e.target.files[0]));
$('#epubGo').addEventListener('click', convertEpub);
$('#fileInput').addEventListener('change', e=>uploadVoice(e.target.files[0]));
$('#fmt').addEventListener('click', e=>{ if(e.target.dataset.v) fmt(e.target.dataset.v); });
$('#exaggeration').addEventListener('input', e=>{ $('#exVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#cfg').addEventListener('input', e=>{ $('#cfgVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#temperature').addEventListener('input', e=>{ $('#tempVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#rep').addEventListener('input', e=>{ $('#repVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#minp').addEventListener('input', e=>{ $('#minpVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#topp').addEventListener('input', e=>{ $('#toppVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#gap').addEventListener('input', e=>{ $('#gapVal').textContent = e.target.value+' ms'; });
$('#chunk').addEventListener('input', e=>{ $('#chunkVal').textContent = e.target.value+' chars'; });
$('#script').addEventListener('input', updateMeters);
$('#goBtn').addEventListener('click', generate);
$('#stopBtn').addEventListener('click', stopJob);
$('#shareBtn').addEventListener('click', share);
// swipe-down on any sheet grabber/head
for(const s of document.querySelectorAll('.sheet')){
  let sy=0, dy=0, drag=false;
  s.addEventListener('touchstart', e=>{ if(e.target.classList.contains('grabber')||e.target.closest('.head')){ drag=true; sy=e.touches[0].clientY; } },{passive:true});
  s.addEventListener('touchmove',  e=>{ if(!drag) return; dy=e.touches[0].clientY-sy; if(dy>0){ s.style.transform=`translateY(${dy}px)`; } },{passive:true});
  s.addEventListener('touchend',   ()=>{ if(!drag) return; drag=false; s.style.transform=''; if(dy>90) closeSheets(); dy=0; });
}

updateMeters(); loadStatus();
if('serviceWorker' in navigator){ navigator.serviceWorker.register('/sw.js').catch(()=>{}); }
</script>
</body></html>"""

MANIFEST_JSON = '''{
  "name": "Chatterbox TTS · GPU",
  "short_name": "Chatterbox",
  "start_url": "/",
  "display": "standalone",
  "background_color": "#0d0b14",
  "theme_color": "#0d0b14",
  "orientation": "portrait",
  "icons": [
    { "src": "/icon.svg", "sizes": "any", "type": "image/svg+xml", "purpose": "any maskable" }
  ]
}'''

ICON_SVG = '''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 512 512">
  <defs><linearGradient id="g" x1="0" y1="0" x2="1" y2="1">
    <stop offset="0" stop-color="#a78bfa"/><stop offset="1" stop-color="#8b5cf6"/></linearGradient></defs>
  <rect width="512" height="512" rx="112" fill="url(#g)"/>
  <text x="50%" y="58%" text-anchor="middle" font-family="-apple-system,Segoe UI,Roboto,sans-serif"
        font-weight="800" font-size="280" fill="#fff">C</text>
</svg>'''

SW_JS = """
const CACHE = "chatterbox-v1";
const SHELL = ["/", "/manifest.webmanifest", "/icon.svg"];
self.addEventListener("install", e => {
  e.waitUntil(caches.open(CACHE).then(c => c.addAll(SHELL)).then(() => self.skipWaiting()));
});
self.addEventListener("activate", e => e.waitUntil(
  caches.keys().then(ks => Promise.all(ks.filter(k => k !== CACHE).map(k => caches.delete(k))))
    .then(() => self.clients.claim())
));
self.addEventListener("fetch", e => {
  const u = new URL(e.request.url);
  if (u.pathname.startsWith("/api/")) return;   // never cache API/SSE
  e.respondWith(
    fetch(e.request).then(r => {
      const copy = r.clone();
      if (r.ok && e.request.method === "GET" && SHELL.includes(u.pathname))
        caches.open(CACHE).then(c => c.put(e.request, copy));
      return r;
    }).catch(() => caches.match(e.request).then(r => r || caches.match("/")))
  );
});
"""

print(f"frontend assets built ({len(INDEX_HTML):,} bytes html)")


### 7️⃣ Backend API · instant
Jobs save output locally before browser delivery and mirror to Google Drive when configured.


In [ ]:
# ── STEP 7 · Backend API (jobs, SSE progress, voice uploads) ─────────────

import asyncio, json, re as _re, uuid, threading, queue as pyqueue, time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from fastapi import FastAPI, HTTPException, Response, Request, UploadFile, File
from fastapi.responses import HTMLResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"],
                   allow_headers=["*"], expose_headers=["X-Meta"])

# ---------------------------------------------------------------------------
# Voice registry — built-in voice + user clips persisted under /kaggle/working
# ---------------------------------------------------------------------------
VOICES_DIR = Path("/kaggle/working/voices")
VOICES_DIR.mkdir(parents=True, exist_ok=True)

def _voice_path(vid: str) -> Path: return VOICES_DIR / f"{vid}.wav"
def _voice_meta(vid: str) -> Path: return VOICES_DIR / f"{vid}.json"

def list_voices() -> list[dict]:
    out = [{"id": "default", "name": "Default voice", "kind": "builtin", "seconds": None}]
    for meta in sorted(VOICES_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime):
        try:
            j = json.loads(meta.read_text())
            vid = meta.stem
            if _voice_path(vid).exists():
                out.append({"id": vid, "name": j.get("name", vid[:8]),
                            "kind": "cloned", "seconds": j.get("seconds")})
        except Exception:
            pass
    return out

@app.post("/api/voices")
async def upload_voice(file: UploadFile):
    raw = await file.read()
    if len(raw) < 1000:        raise HTTPException(400, "file looks empty")
    if len(raw) > 30 << 20:    raise HTTPException(413, "clip too large (30 MB max)")
    vid = uuid.uuid4().hex[:12]
    tmp = VOICES_DIR / f"_up_{vid}"
    try:
        tmp.write_bytes(raw)
        # Transcode anything the phone throws at us (webm/m4a/mp3/…) to 24k mono
        # WAV, capped at 30 s — that's well past the 10 s Chatterbox actually uses.
        from pydub import AudioSegment
        seg = AudioSegment.from_file(str(tmp))
        seg = seg.set_channels(1).set_frame_rate(SAMPLE_RATE)
        if len(seg) > 30_000: seg = seg[:30_000]
        if len(seg) < 1000:   raise HTTPException(400, "clip too short — need at least ~1 s of speech")
        seg.export(str(_voice_path(vid)), format="wav")
        name = _re.sub(r"\s+", " ", Path(file.filename or "my voice").stem).strip()[:60] or "my voice"
        _voice_meta(vid).write_text(json.dumps({"name": name, "seconds": round(len(seg)/1000, 2)}))
    except HTTPException:
        raise
    except Exception as e:
        for p in (_voice_path(vid), _voice_meta(vid)):
            try: p.unlink()
            except Exception: pass
        raise HTTPException(400, f"could not decode audio clip: {e!s}")
    finally:
        try: tmp.unlink()
        except Exception: pass
    # pre-warm the (expensive) conditionals on one GPU eagerly in background —
    # by the time you hit Generate it's usually embedded already on both.
    def _prewarm():
        for m in MODELS.values():
            try:
                with m._cb_lock: ensure_conditionals(m, str(_voice_path(vid)), 0.5)
            except Exception: pass
    threading.Thread(target=_prewarm, daemon=True).start()
    meta = json.loads(_voice_meta(vid).read_text())
    return {"voice": {"id": vid, "name": meta["name"], "kind": "cloned", "seconds": meta["seconds"]}}

@app.delete("/api/voices/{vid}")
def delete_voice(vid: str):
    if not _re.fullmatch(r"[0-9a-f]{12}", vid): raise HTTPException(400, "bad voice id")
    existed = _voice_path(vid).exists()
    for p in (_voice_path(vid), _voice_meta(vid)):
        try: p.unlink()
        except Exception: pass
    for m in MODELS.values():                    # drop cached conds pointing at it
        if str(_voice_path(vid)) == getattr(m, "_cb_voice_path", None):
            try: m.conds = m._cb_default_conds; m._cb_voice_path = None
            except Exception: pass
    if not existed: raise HTTPException(404, "voice not found")
    return {"ok": True}

# ---------------------------------------------------------------------------
# Jobs
# ---------------------------------------------------------------------------
@dataclass
class Job:
    id: str
    text: str; language_id: str; voice_ref: str | None; params: dict; fmt: str
    events: pyqueue.Queue = field(default_factory=pyqueue.Queue)
    audio: bytes | None = None
    mime: str = "audio/wav"
    meta: dict = field(default_factory=dict)
    cancelled: bool = False
    error: str | None = None
    started: float = 0.0

JOBS: dict[str, Job] = {}
EPUB_BOOKS: dict[str, dict] = {}
JOBS_LOCK = threading.Lock()

def _run_job(job: Job):
    try:
        job.started = time.time()
        def prog(done, total):
            if job.cancelled: raise RuntimeError("cancelled")
            job.events.put(("progress", {"done": done, "total": total, "phase": "synthesizing"}))
        sr, wav, nchunks = synth(job.text, job.language_id, job.voice_ref, job.params,
                                 on_progress=prog, is_cancelled=lambda: job.cancelled)
        if job.cancelled: return
        job.events.put(("progress", {"done": nchunks, "total": nchunks, "phase": "encoding"}))
        t_enc0 = time.time()
        audio_bytes, mime = encode_async(sr, wav, job.fmt).result()
        # Persist before notifying the browser. A tunnel/page disconnect cannot
        # waste a completed chapter, and Drive upload runs from the Kaggle VM.
        ext = job.fmt if job.fmt != "opus" else "opus"
        target = OUTPUT_DIR / f"{safe_filename(job.output_name)}.{ext}"
        target.write_bytes(audio_bytes)
        try: drive_id = mirror_to_drive(target)
        except Exception as drive_error:
            drive_id = None; print(f"Drive mirror failed for {target.name}: {drive_error!s}")
        job.audio, job.mime = audio_bytes, mime
        job.meta = {"audio_s": len(wav)/sr, "chunks": nchunks,
                    "wall_s": time.time()-job.started,
                    "encode_s": time.time()-t_enc0,
                    "gpus": len({MODEL_DEVICE[d] for d in WORKERS if MODEL_DEVICE[d].startswith("cuda")}) or 1, "workers": len(WORKERS), "file": target.name, "drive_id": drive_id}
        job.events.put(("done", job.meta))
    except Exception as e:
        if not job.cancelled:
            job.error = str(e); job.events.put(("error", {"message": job.error}))
    finally:
        job.events.put(("__end__", None))

_job_pool = ThreadPoolExecutor(max_workers=4)   # several concurrent jobs share the GPU locks

@app.get("/", response_class=HTMLResponse)
def index():
    return HTMLResponse(INDEX_HTML)

@app.get("/manifest.webmanifest")
def manifest():
    return Response(MANIFEST_JSON, media_type="application/manifest+json")

@app.get("/icon.svg")
def icon():
    return Response(ICON_SVG, media_type="image/svg+xml")

@app.get("/sw.js")
def sw():
    return Response(SW_JS, media_type="application/javascript")

@app.get("/api/status")
def status():
    gpu_names = [g["name"] for g in GPUS] if GPUS else []
    return {
        "model": f"chatterbox-multilingual-{ACTIVE_T3}",
        "watermark": HAS_WATERMARK,
        "n_gpus": len({MODEL_DEVICE[d] for d in WORKERS if MODEL_DEVICE[d].startswith("cuda")}),
        "n_workers": len(WORKERS),
        "n_cpu": N_CPU,
        "gpus": GPUS,
        "fp16": all(FP16.get(d, False) for d in WORKERS if MODEL_DEVICE[d].startswith("cuda")) if GPUS else False,
        "languages": LANGUAGE_LIST,
        "voices": list_voices(),
        "sample_rate": SAMPLE_RATE,
        "accelerator": " + ".join(gpu_names) if gpu_names else "CPU",
    }

def _clamp(v, lo, hi, default):
    try: v = float(v)
    except Exception: return default
    return max(lo, min(hi, v))

@app.post("/api/jobs")
async def submit(req: Request):
    body = await req.json()
    text = (body.get("text") or "").strip()
    if not text: raise HTTPException(400, "text is required")
    if len(text) > 500_000: raise HTTPException(413, "text too large (500k char cap)")
    lang = (body.get("language_id") or DEFAULT_LANG).lower()
    if lang not in LANGUAGES: raise HTTPException(400, f"unknown language: {lang}")
    voice = (body.get("voice") or "default").strip()
    if voice not in ("default", ""):
        if not _re.fullmatch(r"[0-9a-f]{12}", voice) or not _voice_path(voice).exists():
            raise HTTPException(400, "unknown cloned voice — re-upload the clip")
        voice_ref = str(_voice_path(voice))
    else:
        voice_ref = None
    fmt = (body.get("format") or "wav").lower()
    if fmt not in ("wav", "mp3", "flac", "opus"): fmt = "wav"
    params = {
        "exaggeration":       _clamp(body.get("exaggeration", 0.5),      0.0, 2.0,  0.5),
        "cfg_weight":         _clamp(body.get("cfg_weight", 0.5),        0.0, 1.0,  0.5),
        "temperature":        _clamp(body.get("temperature", 0.8),       0.05, 2.0, 0.8),
        "repetition_penalty": _clamp(body.get("repetition_penalty",1.2), 1.0, 5.0,  1.2),
        "min_p":              _clamp(body.get("min_p", 0.05),            0.0, 0.5,  0.05),
        "top_p":              _clamp(body.get("top_p", 1.0),             0.01, 1.0, 1.0),
        "seed":               int(_clamp(body.get("seed", 0),            0, 99999999, 0)),
        "gap_ms":             int(_clamp(body.get("gap_ms", 120),        0, 1500, 120)),
        "chunk_chars":        int(_clamp(body.get("chunk_chars", 260), 140, 400, 260)),
    }
    output_name = safe_filename(body.get("output_name") or "chatterbox")
    job = Job(id=uuid.uuid4().hex, text=text, language_id=lang,
              voice_ref=voice_ref, params=params, fmt=fmt, output_name=output_name)
    with JOBS_LOCK: JOBS[job.id] = job
    _job_pool.submit(_run_job, job)
    return {"job_id": job.id}

@app.post("/api/epub")
async def upload_epub(file: UploadFile = File(...)):
    if not (file.filename or "").lower().endswith(".epub"): raise HTTPException(400, "upload an .epub file")
    eid = uuid.uuid4().hex; path = EPUB_DIR / f"{eid}.epub"; path.write_bytes(await file.read())
    try: chapters = epub_chapters(path)
    except Exception as e: path.unlink(missing_ok=True); raise HTTPException(400, f"could not read EPUB: {e!s}")
    EPUB_BOOKS[eid] = {"path": path, "chapters": chapters}
    return {"epub_id": eid, "chapters": [{"id": c["id"], "title": c["title"], "chars": len(c["text"])} for c in chapters]}

@app.post("/api/epub/{epub_id}/jobs")
async def epub_jobs(epub_id: str, request: Request):
    # Browser can submit selected IDs; it receives normal job IDs, one per title.
    book = EPUB_BOOKS.get(epub_id)
    if not book: raise HTTPException(404, "EPUB expired; upload it again")
    body = await request.json(); wanted=set(body.get("chapter_ids") or [])
    selected=[c for c in book["chapters"] if c["id"] in wanted]
    if not selected: raise HTTPException(400, "select at least one chapter")
    jobs=[]
    for c in selected:
        # Reuse validated normal-job route so EPUBs get identical TTS settings.
        body2=dict(body); body2.update({"text": c["text"], "output_name": c["title"]})
        class _R:
            async def json(self): return body2
        result=await submit(_R()); jobs.append({"title":c["title"], **result})
    return {"jobs": jobs, "saved_to": str(OUTPUT_DIR), "drive_enabled": bool(DRIVE_SERVICE)}

@app.get("/api/jobs/{job_id}/events")
async def events(job_id: str, request: Request):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "job not found")
    async def gen():
        loop = asyncio.get_event_loop()
        yield "retry: 3000\n\n"
        while True:
            if await request.is_disconnected(): break
            try:
                evt, data = await loop.run_in_executor(None, lambda: job.events.get(timeout=1.0))
            except Exception:
                yield ": keepalive\n\n"; continue
            if evt == "__end__": break
            yield f"event: {evt}\ndata: {json.dumps(data)}\n\n"
    headers = {"Cache-Control": "no-cache", "X-Accel-Buffering": "no", "Connection": "keep-alive"}
    return StreamingResponse(gen(), media_type="text/event-stream", headers=headers)

@app.get("/api/jobs/{job_id}/audio")
def get_audio(job_id: str):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "job not found")
    if job.error: raise HTTPException(500, job.error)
    if job.audio is None: raise HTTPException(425, "not ready")
    hdr = {"X-Meta": json.dumps(job.meta), "Cache-Control": "no-store",
           "Content-Disposition": f'attachment; filename="chatterbox.{job.fmt}"'}
    return Response(job.audio, media_type=job.mime, headers=hdr)

@app.delete("/api/jobs/{job_id}")
def cancel(job_id: str):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "job not found")
    job.cancelled = True
    return {"ok": True}

def _reap():
    while True:
        time.sleep(60)
        with JOBS_LOCK:
            if len(JOBS) > 32:
                old = sorted(JOBS.values(), key=lambda j: j.started)[: len(JOBS)-32]
                for j in old: JOBS.pop(j.id, None)
if not globals().get("_reaper_started"):             # cell re-run: don't stack reapers
    globals()["_reaper_started"] = True
    threading.Thread(target=_reap, daemon=True).start()

print("FastAPI app defined")


### 8️⃣ Start server + tunnel · ~10 s → **your URL appears here**

Uvicorn on `0.0.0.0:7860` + a Cloudflare quick tunnel. Watch this cell's output and **tap the big
`trycloudflare.com` link** — that's your app. Keep this Kaggle tab open while you use it.


In [ ]:
# ── STEP 8 · Start server + tunnel → tap the big URL in the output ──────

import threading, uvicorn, subprocess, re, time, sys

PORT = 7860
_server = None
def _serve():
    global _server
    cfg = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning", access_log=False)
    _server = uvicorn.Server(cfg)
    _server.run()

# Re-run friendly: only one uvicorn + one tunnel, ever.
if not globals().get("_server_started"):
    threading.Thread(target=_serve, daemon=True).start()
    globals()["_server_started"] = True
    time.sleep(2)
print(f"uvicorn on http://0.0.0.0:{PORT}")

URL_RE = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
_prev = globals().get("_cf_proc")

if _prev is not None and _prev.poll() is None and globals().get("_public_url"):
    public_url = globals()["_public_url"]            # tunnel still alive — reuse it
    print(f"tunnel already up: {public_url}")
else:
    # Start cloudflared quick tunnel; scrape stdout for the public URL
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    globals()["_cf_proc"] = cf_proc
    public_url = None
    t_start = time.time()
    while time.time() - t_start < 60:
        line = cf_proc.stdout.readline()
        if not line: time.sleep(0.1); continue
        sys.stdout.write(line)
        m = URL_RE.search(line)
        if m: public_url = m.group(0); break
    globals()["_public_url"] = public_url

if public_url:
    print("\n" + "="*60)
    print(f"  📱  Open this on your phone:  {public_url}")
    print(f"      (also http://localhost:{PORT} from inside the kernel)")
    print("="*60)
    try:
        from IPython.display import display, HTML
        display(HTML(
            f'<div style="padding:14px;border-radius:16px;background:#0d0b14;border:1px solid #2b2540;'
            f'font-family:-apple-system,system-ui,sans-serif;text-align:center">'
            f'<div style="font-size:12px;color:#a49eb8;margin-bottom:10px">✅ Your app is live — tap to open</div>'
            f'<a href="{public_url}" target="_blank" style="display:block;padding:16px 18px;border-radius:14px;'
            f'background:linear-gradient(135deg,#a78bfa,#8b5cf6);color:#fff;text-decoration:none;'
            f'font-size:18px;font-weight:800">📲 Open Chatterbox TTS</a>'
            f'<div style="margin-top:10px;font-size:11px;color:#a49eb8;word-break:break-all">{public_url}</div>'
            f'</div>'))
    except Exception:
        pass
else:
    print("cloudflared did not report a URL — check its output above.")


### 9️⃣ (Optional) Benchmark — one GPU vs the pair · ~2 min

Same seed, same chunks: single replica vs both GPUs, plus a warm rerun. Expect ~**2× wall-clock win** on the long
text; safe to skip on a fresh session.


In [ ]:
# ── STEP 9 · (Optional) benchmark: one GPU vs both ──────────────────────

# Benchmark: prove the dual-GPU fan-out pays off. Fixed seed → identical chunks
# on both paths, so the delta is purely scheduling (and fp16, already applied).
import time

PARAMS = dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8,
              repetition_penalty=1.2, min_p=0.05, top_p=1.0,
              seed=42, gap_ms=120, chunk_chars=260)

BENCH_SHORT = ("Chatterbox makes short work of a quick hello. " * 2).strip()
BENCH_LONG  = ("The two graphics cards share every sentence between them, "
               "and the first one to finish simply takes the next chunk. " * 8).strip()

for name, txt in [("short", BENCH_SHORT), ("long", BENCH_LONG)]:
    t0 = time.time()
    _, w1, n = synth(txt, "en", None, dict(PARAMS), devices=DEVICES[:1])
    t_one = time.time() - t0
    t0 = time.time()
    _, wN, n = synth(txt, "en", None, dict(PARAMS))                    # all devices
    t_all = time.time() - t0
    print(f"{name:6s} {n:3d} chunks  1 gpu={t_one:6.2f}s   {len(DEVICES)} gpu={t_all:6.2f}s"
          f"   ->  {t_one/t_all:4.2f}x   ({len(wN)/SAMPLE_RATE:.1f}s audio)")

t0 = time.time()
_, w2, _ = synth(BENCH_LONG, "en", None, dict(PARAMS))
print(f"long (warm rerun, conds cached): {time.time()-t0:6.2f}s")
